# 04 — Policy Fairness

**Task.** Given two organizational policy descriptions, predict which one received the majority vote as fairer from human raters. Binary classification, "first" vs. "second". The score is accuracy.

**What the winners got.** PAID .828 · Akben .828 (tied) · Wonderlic .792 · Hungry Llama .760.

**The pattern.** This is the most-converged task of the four. Everyone used full-train-set few-shot with GPT-4 or Mixtral. The differences are small and mostly attributable to model choice.

## Setup

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import json
from src.adapters import FairnessAdapter
from src.harness import Harness, CallSpec, mode_reducer
from src.scoring import accuracy
from src.run import load_csv, DATA_DIR

print(f"Data directory: {DATA_DIR.resolve()}")
print(f"Fairness inputs (train): {(DATA_DIR / 'fairness_train.csv').exists()}")
print(f"Fairness inputs (dev):   {(DATA_DIR / 'fairness_val_public.csv').exists() or (DATA_DIR / 'fairness_dev_inputs.csv').exists()}")
print(f"Fairness inputs (test):  {(DATA_DIR / 'fairness_test_public.csv').exists() or (DATA_DIR / 'fairness_test_inputs.csv').exists()}")

## The data

Each row contains two organizational policies that address the same workplace situation (e.g., conflict resolution, scheduling flexibility, performance reviews). Human raters voted on which one struck them as fairer. The training set is small (~25 paired comparisons in the original release), which is why every winning team used the entire training set as few-shot context.

Below is what a comparison looks like, from the Hungry Llama deck:

> **Option 1:** Conflict Resolution Workshops — We offer workshops where employees learn dispute resolution and active listening...
>
> **Option 2:** Conflict Resolution Workbooks — Resources to help employees self-resolve conflicts on their own time...

Both are reasonable. The "fairer" choice depends on how raters weigh organizational support vs. employee autonomy, and that judgment varies. The base rate of "first" being chosen is about 57%.

In [ ]:
fairness_train = load_csv(DATA_DIR / "fairness_train.csv")
if fairness_train:
    print(f"Loaded {len(fairness_train)} training pairs")
    print(f"\nColumns: {list(fairness_train[0].keys())}")
    print(f"\nLabel distribution:")
    from collections import Counter
    print(Counter(r["majority_vote"] for r in fairness_train))
    print(f"\nFirst training pair:")
    r = fairness_train[0]
    print(f"  Option 1: {r['first_option'][:200]}")
    print(f"  Option 2: {r['second_option'][:200]}")
    print(f"  Majority vote: {r['majority_vote']}")
else:
    print("Fairness training data not present.")
    print("Synthetic pair for prompt inspection:")
    fairness_train = [{
        "_id": "sample1",
        "first_option": "We offer conflict resolution workshops where employees learn dispute resolution and active listening skills.",
        "second_option": "Resources include conflict resolution workbooks employees can use on their own time.",
        "majority_vote": "first",
    }]

## All four teams' approaches converge here

| Team | Model | Few-shot | Self-consistency | Score |
|------|-------|----------|-------------------|-------|
| PAID | GPT-4 | All 24 train rows + auto-reasons | No | .828 |
| Akben | GPT-4 | All 24 train rows | Yes (N=5) | .828 |
| Hungry Llama | Mixtral 8x7B | All 23 train rows | No | .760 |
| Wonderlic | (mixed) | (varies) | No | .792 |

The PAID-vs-Akben tie at .828 is interesting: same model, same training data, same in-context learning approach. Akben added self-consistency, PAID added auto-reasons. Both moves gave the same accuracy ceiling. With a small test set (~25-30 rows in test for fairness), there's also non-trivial measurement noise — the next decimal place might come down to which specific rows each team got right.

The 7-point gap between PAID/Akben and Hungry Llama reflects the model: Mixtral 8x7B (open weights) vs. GPT-4. There's no architectural innovation that closes a gap that size.

## The unified-harness approach

The FairnessAdapter uses essentially the same approach as PAID/Akben: full training set as few-shot, structured-output JSON for the "first"/"second" choice, optional self-consistency via the harness wrapper.

The only difference from PAID is the auto-reasons step (we skip it; the benefit is small relative to the cost). The only difference from Akben is the self-consistency wrapper (off by default; turn it on via `--self-consistency 5`).

In [ ]:
adapter = FairnessAdapter()
print(f"Adapter: {adapter.task_name}")
print(f"K few-shot examples: {adapter.k_examples}")
print(f"Response format: strict JSON with choice ∈ {{'first', 'second'}}")
print()

sample_test = {
    "first_option": "Performance reviews are conducted annually by direct managers with peer input.",
    "second_option": "Performance reviews are conducted quarterly by a panel including HR and rotating peers.",
}
messages = adapter.build_messages(sample_test, fairness_train[:3])
print(f"Number of messages: {len(messages)}")
print(f"\n--- System ---\n{messages[0]['content']}")
print(f"\n--- First few-shot user ---\n{messages[1]['content'][:300]}...")
print(f"\n--- First few-shot assistant ---\n{messages[2]['content']}")
print(f"\n--- Final user (test row) ---\n{messages[-1]['content'][:300]}...")

## End-to-end run

In [ ]:
from src.run import run_task

# Single run, default settings
result = run_task(
    task="fairness",
    split="dev",
    model="gpt-4o-2024-08-06",
    self_consistency=1,
    output_path=None,
    row_id=None,
    similarity_examples=False,
)
if result["status"] == "ok":
    print(f"Fairness dev: n={result['n']}, accuracy={result.get('score', 'n/a'):.4f}")
else:
    print(f"Status: {result['status']}")
    print(f"Message: {result.get('message')}")

In [ ]:
# Akben-style self-consistency variant
result_sc = run_task(
    task="fairness",
    split="dev",
    model="gpt-4o-2024-08-06",
    self_consistency=5,  # 5 calls per test row at T=0.7, mode
    output_path=None,
    row_id=None,
    similarity_examples=False,
)
if result_sc["status"] == "ok":
    print(f"Fairness dev (N=5 SC): n={result_sc['n']}, accuracy={result_sc.get('score', 'n/a'):.4f}")

## Discussion — when the field converges

Three things to take from the fact that all four teams ended up doing almost the same thing on fairness:

1. **When training data is small, few-shot is just "use it all."** With 24-25 paired examples, there's no benefit to K-of-N selection — you can fit them all in context.

2. **Self-consistency doesn't help when you're already at the model's accuracy ceiling.** PAID/Akben both at .828 implies GPT-4 has roughly 17% of test rows where the right answer is genuinely ambiguous to it. Self-consistency only helps when the model's uncertainty resolves *toward* the right answer; if the uncertainty is genuine, no number of repeated calls fixes it.

3. **Structured outputs are pure free wins.** The PAID/Akben notebooks both had defensive parse logic for cases where GPT-4 returned "first" instead of `{"choice": "first"}` or "Option 1" or "I think the first one." Strict JSON mode eliminates this class of error entirely. If the 2024 winners had this in 2024, they probably would have scored .835-.85 instead of .828.

A pedagogical point worth dwelling on: when four very different teams independently land at the same approach, that's strong evidence the approach is near-optimal for the task. Don't waste cycles trying to be clever — match the convergence point, focus your innovation on the harder tasks (clarity, interview).